In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
pip install lightgbm wandb transformers

Note: you may need to restart the kernel to use updated packages.


In [3]:
"""
============================================================
SMART MCQ SOLVER CHALLENGE — MODEL 1: TF-IDF + LightGBM
============================================================
Classical ML Baseline (Milestone 2 requirement)
Metric: MAP@3 (Mean Average Precision at 3)

Run this cell/script FIRST — no GPU needed.
============================================================
"""

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

# ── W&B Setup ──
import wandb
# Use Kaggle Secrets (add WANDB_API_KEY in Add-ons → Secrets)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    wandb_key = secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_key)
except Exception:
    wandb.login()  # fallback: interactive prompt

# ── MAP@3 Evaluation ──
def map_at_3(y_true, y_pred_top3):
    """Compute Mean Average Precision @ 3."""
    scores = []
    for true, preds in zip(y_true, y_pred_top3):
        score = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return np.mean(scores)

# ── Data Loading ──
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"  # Kaggle path
if not os.path.exists(DATA_DIR):
    DATA_DIR = "."  # local fallback

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Answer distribution:\n{train['answer'].value_counts()}")

LABELS = ['A', 'B', 'C', 'D', 'E']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}

# ── Feature Engineering ──
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

def build_tfidf_features(df, prompt_vec=None, option_vecs=None, fit=True):
    """
    Create TF-IDF features by:
    1. TF-IDF on prompts
    2. TF-IDF on each option text
    3. Cosine similarity between prompt and each option (shared vocab)
    4. Text length features
    """
    if fit:
        prompt_vec = TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True)
        prompt_tfidf = prompt_vec.fit_transform(df['prompt'].fillna(''))
        option_vecs = {}
        option_tfidfs = {}
        for col in LABELS:
            option_vecs[col] = TfidfVectorizer(max_features=3000, ngram_range=(1,2), sublinear_tf=True)
            option_tfidfs[col] = option_vecs[col].fit_transform(df[col].fillna(''))
        # Shared vectorizer for similarity (same vocab for prompt & options)
        all_text = pd.concat([df['prompt'].fillna('')] + [df[c].fillna('') for c in LABELS])
        shared_vec = TfidfVectorizer(max_features=8000, ngram_range=(1,2), sublinear_tf=True)
        shared_vec.fit(all_text)
        option_vecs['__shared__'] = shared_vec
    else:
        prompt_tfidf = prompt_vec.transform(df['prompt'].fillna(''))
        option_tfidfs = {}
        for col in LABELS:
            option_tfidfs[col] = option_vecs[col].transform(df[col].fillna(''))

    # Length features
    len_feats = np.column_stack([
        df['prompt'].fillna('').str.len().values,
        *[df[c].fillna('').str.len().values for c in LABELS],
        *[df[c].fillna('').str.split().str.len().fillna(0).values for c in LABELS],
    ])

    # Similarity features using SHARED vectorizer (same dimensions for prompt & option)
    shared_vec = option_vecs['__shared__']
    prompt_shared = shared_vec.transform(df['prompt'].fillna(''))
    sim_feats = np.column_stack([
        np.array(prompt_shared.multiply(
            shared_vec.transform(df[c].fillna(''))
        ).sum(axis=1)).flatten()
        for c in LABELS
    ])

    # Combine: all option TF-IDFs + length + similarity
    combined = hstack([prompt_tfidf] + [option_tfidfs[c] for c in LABELS])

    # Add dense features
    dense = np.hstack([len_feats, sim_feats])

    from scipy.sparse import csr_matrix
    combined = hstack([combined, csr_matrix(dense)])

    return combined, prompt_vec, option_vecs

X_train, prompt_vec, option_vecs = build_tfidf_features(train, fit=True)
X_test, _, _ = build_tfidf_features(test, prompt_vec, option_vecs, fit=False)
y_train = train['answer'].map(label2id).values
print(f"Features: X_train={X_train.shape}, X_test={X_test.shape}")

# ── Cross-Validation ──
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_probs = np.zeros((len(train), 5))
test_probs = np.zeros((len(test), 5))

run = wandb.init(
    project="smart-mcq-solver",
    name="model1-tfidf-lgbm",
    config={"model": "LightGBM", "features": "TF-IDF", "n_folds": N_FOLDS}
)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\n--- Fold {fold+1}/{N_FOLDS} ---")
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    model = lgb.LGBMClassifier(
        n_estimators=1000, learning_rate=0.05, num_leaves=63,
        max_depth=8, subsample=0.8, colsample_bytree=0.5,
        reg_alpha=0.1, reg_lambda=1.0, random_state=42+fold,
        n_jobs=-1, verbose=-1
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
    )

    val_probs = model.predict_proba(X_val)
    oof_probs[val_idx] = val_probs
    test_probs += model.predict_proba(X_test) / N_FOLDS

    # Fold MAP@3
    val_top3 = [
        [id2label[i] for i in np.argsort(-p)[:3]] for p in val_probs
    ]
    val_true = [id2label[y] for y in y_val]
    fold_map3 = map_at_3(val_true, val_top3)
    print(f"Fold {fold+1} MAP@3: {fold_map3:.4f}")
    wandb.log({"fold": fold+1, "fold_map3": fold_map3})

# Overall OOF MAP@3
oof_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in oof_probs]
oof_true = [id2label[y] for y in y_train]
overall_map3 = map_at_3(oof_true, oof_top3)
print(f"\n=== Model 1 (LightGBM) Overall OOF MAP@3: {overall_map3:.4f} ===")
wandb.log({"overall_map3": overall_map3})
wandb.finish()

# ── Save Model 1 Predictions ──
np.save("lgbm_test_probs.npy", test_probs)
np.save("lgbm_oof_probs.npy", oof_probs)

# ── Generate Model 1 Submission ──
test_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in test_probs]
sub = pd.DataFrame({
    "ID": test["id"],
    "Prediction": [" ".join(t) for t in test_top3]
})
sub.to_csv("submission_lgbm.csv", index=False)
print("Model 1 submission saved to submission_lgbm.csv")
print(sub.head(10))


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 24f1002825 (24f1002825-iit-madras-foundation) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Train: (2000, 8), Test: (500, 7)
Answer distribution:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Features: X_train=(2000, 17964), X_test=(500, 17964)


wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260605_121009-c26l4d5f
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model1-tfidf-lgbm
wandb: ⭐️ View project at https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver/runs/c26l4d5f



--- Fold 1/5 ---
Training until validation scores don't improve for 50 rounds
[100]	valid_0's multi_logloss: 0.0250038
[200]	valid_0's multi_logloss: 0.0057765
[300]	valid_0's multi_logloss: 0.00410418
[400]	valid_0's multi_logloss: 0.00388996
[500]	valid_0's multi_logloss: 0.00386324
[600]	valid_0's multi_logloss: 0.00385501
[700]	valid_0's multi_logloss: 0.00385183
[800]	valid_0's multi_logloss: 0.00385042
[900]	valid_0's multi_logloss: 0.00384786
[1000]	valid_0's multi_logloss: 0.00384675
Did not meet early stopping. Best iteration is:
[1000]	valid_0's multi_logloss: 0.00384675
Fold 1 MAP@3: 1.0000

--- Fold 2/5 ---
Training until validation scores don't improve for 50 rounds
[100]	valid_0's multi_logloss: 0.0243007
[200]	valid_0's multi_logloss: 0.00645413
[300]	valid_0's multi_logloss: 0.00466549
[400]	valid_0's multi_logloss: 0.00439445
[500]	valid_0's multi_logloss: 0.00436358
[600]	valid_0's multi_logloss: 0.00435029
[700]	valid_0's multi_logloss: 0.00434433
[800]	valid_0's mu

wandb: updating run metadata


Fold 5 MAP@3: 1.0000

=== Model 1 (LightGBM) Overall OOF MAP@3: 1.0000 ===


wandb: uploading history steps 4-5, summary, console lines 66-79
wandb: 
wandb: Run history:
wandb:         fold ▁▃▅▆█
wandb:    fold_map3 ▁▁▁▁▁
wandb: overall_map3 ▁
wandb: 
wandb: Run summary:
wandb:         fold 5
wandb:    fold_map3 1
wandb: overall_map3 1
wandb: 
wandb: 🚀 View run model1-tfidf-lgbm at: https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver/runs/c26l4d5f
wandb: ⭐️ View project at: https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260605_121009-c26l4d5f/logs


Model 1 submission saved to submission_lgbm.csv
   ID Prediction
0   1      A C E
1   2      B D C
2   3      B E A
3   4      E C A
4   5      C D A
5   6      D C B
6   7      E C D
7   8      B C A
8   9      C B A
9  10      B D C


In [4]:
"""
============================================================
SMART MCQ SOLVER CHALLENGE — MODEL 2: PyTorch Text CNN
============================================================
Neural Network built FROM SCRATCH (Milestone 3 requirement)
Uses pre-trained GloVe embeddings but custom CNN architecture.
Metric: MAP@3

Requires GPU. Run on Kaggle with GPU accelerator.
============================================================
"""

import pandas as pd
import numpy as np
import os, gc, warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
import wandb

# ── Reproducibility ──
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── MAP@3 ──
def map_at_3(y_true, y_pred_top3):
    scores = []
    for true, preds in zip(y_true, y_pred_top3):
        score = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return np.mean(scores)

# ── Data ──
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "."

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
LABELS = ['A', 'B', 'C', 'D', 'E']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}

# ── Tokenizer (simple word-level) ──
import re
from collections import Counter

def tokenize(text):
    """Simple whitespace + punctuation tokenizer."""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.split()

# Build vocabulary from train + test
all_texts = list(train['prompt']) + [train[c].tolist() for c in LABELS]
all_texts_flat = list(train['prompt']) + list(test['prompt'])
for c in LABELS:
    all_texts_flat += list(train[c]) + list(test[c])

word_counts = Counter()
for text in all_texts_flat:
    word_counts.update(tokenize(str(text)))

# Keep top 50K words
VOCAB_SIZE = 50000
vocab = {'<PAD>': 0, '<UNK>': 1}
for word, _ in word_counts.most_common(VOCAB_SIZE):
    vocab[word] = len(vocab)
print(f"Vocabulary size: {len(vocab)}")

def encode_text(text, max_len=128):
    tokens = tokenize(str(text))[:max_len]
    ids = [vocab.get(t, 1) for t in tokens]
    # Pad
    ids = ids + [0] * (max_len - len(ids))
    return ids

# ── Load GloVe Embeddings ──
GLOVE_DIM = 100
GLOVE_PATH = "/kaggle/input/glove-global-vectors-for-word-representation/glove.6B.100d.txt"
if not os.path.exists(GLOVE_PATH):
    GLOVE_PATH = "glove.6B.100d.txt"  # local fallback

embedding_matrix = np.random.normal(0, 0.1, (len(vocab), GLOVE_DIM)).astype(np.float32)
embedding_matrix[0] = 0  # PAD = zeros

if os.path.exists(GLOVE_PATH):
    print("Loading GloVe embeddings...")
    with open(GLOVE_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip().split(' ')
            word = parts[0]
            if word in vocab:
                embedding_matrix[vocab[word]] = np.array(parts[1:], dtype=np.float32)
    print("GloVe loaded.")
else:
    print("WARNING: GloVe not found, using random embeddings.")
    print("Add 'glove-global-vectors-for-word-representation' as a Kaggle dataset.")

# ── Dataset ──
MAX_PROMPT_LEN = 80
MAX_OPTION_LEN = 100

class MCQDataset(Dataset):
    def __init__(self, df, is_test=False):
        self.df = df.reset_index(drop=True)
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt_ids = encode_text(row['prompt'], MAX_PROMPT_LEN)
        option_ids = [encode_text(row[c], MAX_OPTION_LEN) for c in LABELS]

        prompt_t = torch.tensor(prompt_ids, dtype=torch.long)
        options_t = torch.tensor(option_ids, dtype=torch.long)  # (5, max_option_len)

        if self.is_test:
            return prompt_t, options_t
        else:
            label = label2id[row['answer']]
            return prompt_t, options_t, torch.tensor(label, dtype=torch.long)

# ── Text CNN Model (from scratch) ──
class TextCNNMCQ(nn.Module):
    """
    Custom Text CNN for Multiple Choice QA.
    Architecture:
    - Shared embedding layer (pre-trained GloVe)
    - 1D CNN with multiple kernel sizes over prompt and each option
    - Concatenate prompt + option representations
    - FC layers to produce a score per option
    """
    def __init__(self, pretrained_embeddings, kernel_sizes=[2, 3, 4, 5],
                 num_filters=128, hidden_dim=256, dropout=0.3):
        super().__init__()
        vocab_size, embed_dim = pretrained_embeddings.shape

        # Embedding (frozen GloVe + trainable fine-tune layer)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding.weight = nn.Parameter(
            torch.tensor(pretrained_embeddings), requires_grad=False
        )
        self.embed_proj = nn.Linear(embed_dim, embed_dim)  # learnable projection

        # Multi-kernel CNN
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, k, padding=k//2)
            for k in kernel_sizes
        ])
        self.conv_bn = nn.ModuleList([
            nn.BatchNorm1d(num_filters) for _ in kernel_sizes
        ])

        cnn_out = num_filters * len(kernel_sizes)

        # Classifier: prompt_feats + option_feats → score
        self.fc = nn.Sequential(
            nn.Linear(cnn_out * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.BatchNorm1d(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def encode_text(self, x):
        """x: (batch, seq_len) → (batch, cnn_out)"""
        emb = self.embedding(x)          # (B, L, D)
        emb = self.embed_proj(emb)       # (B, L, D)
        emb = emb.permute(0, 2, 1)      # (B, D, L)

        conv_outs = []
        for conv, bn in zip(self.convs, self.conv_bn):
            c = F.relu(bn(conv(emb)))    # (B, F, L')
            c = F.adaptive_max_pool1d(c, 1).squeeze(-1)  # (B, F)
            conv_outs.append(c)

        return torch.cat(conv_outs, dim=1)  # (B, cnn_out)

    def forward(self, prompt, options):
        """
        prompt:  (B, prompt_len)
        options: (B, 5, option_len)
        returns: (B, 5) logits
        """
        B = prompt.size(0)
        prompt_feat = self.encode_text(prompt)  # (B, cnn_out)

        logits = []
        for i in range(5):
            opt_feat = self.encode_text(options[:, i, :])  # (B, cnn_out)
            combined = torch.cat([prompt_feat, opt_feat], dim=1)  # (B, 2*cnn_out)
            score = self.fc(combined)  # (B, 1)
            logits.append(score)

        return torch.cat(logits, dim=1)  # (B, 5)

# ── Training ──
N_FOLDS = 5
EPOCHS = 15
BATCH_SIZE = 32
LR = 1e-3

# Use Kaggle Secrets (add WANDB_API_KEY in Add-ons → Secrets)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    wandb_key = secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_key)
except Exception:
    wandb.login()  # fallback: interactive prompt

wandb.init(project="smart-mcq-solver", name="model2-textcnn",
           config={"model": "TextCNN", "epochs": EPOCHS, "lr": LR, "batch_size": BATCH_SIZE})

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
y_all = train['answer'].map(label2id).values
oof_probs = np.zeros((len(train), 5))
test_probs = np.zeros((len(test), 5))
test_ds = MCQDataset(test, is_test=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

for fold, (tr_idx, val_idx) in enumerate(skf.split(train, y_all)):
    print(f"\n{'='*50}")
    print(f"Fold {fold+1}/{N_FOLDS}")
    print(f"{'='*50}")

    train_ds = MCQDataset(train.iloc[tr_idx])
    val_ds   = MCQDataset(train.iloc[val_idx])
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = TextCNNMCQ(embedding_matrix).to(DEVICE)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss()

    best_map3 = 0
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for prompt, options, labels in train_loader:
            prompt, options, labels = prompt.to(DEVICE), options.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(prompt, options)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
        scheduler.step()

        # Validation
        model.eval()
        val_preds_all = []
        val_labels_all = []
        with torch.no_grad():
            for prompt, options, labels in val_loader:
                prompt, options = prompt.to(DEVICE), options.to(DEVICE)
                logits = model(prompt, options)
                probs = F.softmax(logits, dim=1).cpu().numpy()
                val_preds_all.append(probs)
                val_labels_all.append(labels.numpy())

        val_preds_all = np.concatenate(val_preds_all)
        val_labels_all = np.concatenate(val_labels_all)

        val_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in val_preds_all]
        val_true = [id2label[y] for y in val_labels_all]
        epoch_map3 = map_at_3(val_true, val_top3)

        avg_loss = train_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val MAP@3: {epoch_map3:.4f}")
        wandb.log({"fold": fold+1, "epoch": epoch+1, "train_loss": avg_loss, "val_map3": epoch_map3})

        if epoch_map3 > best_map3:
            best_map3 = epoch_map3
            best_val_preds = val_preds_all.copy()
            # Save best model state for test inference
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(f"Fold {fold+1} Best MAP@3: {best_map3:.4f}")
    oof_probs[val_idx] = best_val_preds

    # Test inference with best model
    model.load_state_dict(best_state)
    model.to(DEVICE).eval()
    fold_test_preds = []
    with torch.no_grad():
        for prompt, options in test_loader:
            prompt, options = prompt.to(DEVICE), options.to(DEVICE)
            logits = model(prompt, options)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            fold_test_preds.append(probs)
    test_probs += np.concatenate(fold_test_preds) / N_FOLDS

    del model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()

# Overall OOF MAP@3
oof_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in oof_probs]
oof_true = [id2label[y] for y in y_all]
overall_map3 = map_at_3(oof_true, oof_top3)
print(f"\n=== Model 2 (TextCNN) Overall OOF MAP@3: {overall_map3:.4f} ===")
wandb.log({"overall_map3": overall_map3})
wandb.finish()

np.save("textcnn_test_probs.npy", test_probs)
np.save("textcnn_oof_probs.npy", oof_probs)

# Submission
test_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in test_probs]
sub = pd.DataFrame({
    "ID": test["id"],
    "Prediction": [" ".join(t) for t in test_top3]
})
sub.to_csv("submission_textcnn.csv", index=False)
print("Model 2 submission saved to submission_textcnn.csv")
print(sub.head(10))


Device: cuda
Vocabulary size: 2975
Add 'glove-global-vectors-for-word-representation' as a Kaggle dataset.


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260605_121111-eba6lpy9
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model2-textcnn
wandb: ⭐️ View project at https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver/runs/eba6lpy9



Fold 1/5
Epoch 1/15 | Loss: 1.5182 | Val MAP@3: 0.9396
Epoch 2/15 | Loss: 0.5417 | Val MAP@3: 0.9988
Epoch 3/15 | Loss: 0.1670 | Val MAP@3: 1.0000
Epoch 4/15 | Loss: 0.1033 | Val MAP@3: 0.9983
Epoch 5/15 | Loss: 0.0821 | Val MAP@3: 1.0000
Epoch 6/15 | Loss: 0.0748 | Val MAP@3: 1.0000
Epoch 7/15 | Loss: 0.0401 | Val MAP@3: 1.0000
Epoch 8/15 | Loss: 0.0205 | Val MAP@3: 1.0000
Epoch 9/15 | Loss: 0.0294 | Val MAP@3: 1.0000
Epoch 10/15 | Loss: 0.0171 | Val MAP@3: 1.0000
Epoch 11/15 | Loss: 0.0129 | Val MAP@3: 1.0000
Epoch 12/15 | Loss: 0.0095 | Val MAP@3: 1.0000
Epoch 13/15 | Loss: 0.0083 | Val MAP@3: 1.0000
Epoch 14/15 | Loss: 0.0069 | Val MAP@3: 1.0000
Epoch 15/15 | Loss: 0.0076 | Val MAP@3: 1.0000
Fold 1 Best MAP@3: 1.0000

Fold 2/5
Epoch 1/15 | Loss: 1.4922 | Val MAP@3: 0.9179
Epoch 2/15 | Loss: 0.6579 | Val MAP@3: 1.0000
Epoch 3/15 | Loss: 0.2142 | Val MAP@3: 1.0000
Epoch 4/15 | Loss: 0.1447 | Val MAP@3: 1.0000
Epoch 5/15 | Loss: 0.0653 | Val MAP@3: 1.0000
Epoch 6/15 | Loss: 0.1005 | 

wandb: updating run metadata



=== Model 2 (TextCNN) Overall OOF MAP@3: 1.0000 ===


wandb: uploading history steps 65-75, summary, console lines 89-101; uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:        epoch ▁▁▃▄▅▅▇▁▁▂▄▅▅▅▇▁▂▃▃▄▆▇▇█▁▃▄▅▅▆▇█▁▁▂▃▄▅▅█
wandb:         fold ▁▁▁▁▁▁▁▁▁▃▃▃▃▃▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆██████████
wandb: overall_map3 ▁
wandb:   train_loss █▃▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▄▂▂▁▁▁▁▁▃▂▁▁▁▁▁▁█▄▁▁▁▁▁
wandb:     val_map3 ▃███████▁█████████████▅████████▁████████
wandb: 
wandb: Run summary:
wandb:        epoch 15
wandb:         fold 5
wandb: overall_map3 1
wandb:   train_loss 0.00848
wandb:     val_map3 1
wandb: 
wandb: 🚀 View run model2-textcnn at: https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver/runs/eba6lpy9
wandb: ⭐️ View project at: https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260605_121111-eba6lpy9/logs


Model 2 submission saved to submission_textcnn.csv
   ID Prediction
0   1      A E C
1   2      B D A
2   3      B E D
3   4      E C A
4   5      C A D
5   6      D B C
6   7      E C D
7   8      B E A
8   9      C D E
9  10      B C D


In [5]:
"""
============================================================
SMART MCQ SOLVER CHALLENGE — MODEL 3: DeBERTa-v3 MCQA
============================================================
Pre-trained Transformer fine-tuned for MCQA (Milestone 5)
This is the PRIMARY model for maximum MAP@3 score.

MUST run on Kaggle with GPU (T4/P100).
Add 'deberta-v3-base' as a Kaggle dataset if offline.
============================================================
"""

import pandas as pd
import numpy as np
import os, gc, warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
import wandb

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── MAP@3 ──
def map_at_3(y_true, y_pred_top3):
    scores = []
    for true, preds in zip(y_true, y_pred_top3):
        score = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return np.mean(scores)

# ── Data ──
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "."

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
LABELS = ['A', 'B', 'C', 'D', 'E']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}

# ── Model Path ──
# Try Kaggle offline dataset first, then HuggingFace online
MODEL_NAME = "microsoft/deberta-v3-base"
KAGGLE_MODEL_PATH = "/kaggle/input/deberta-v3-base"
if os.path.exists(KAGGLE_MODEL_PATH):
    MODEL_NAME = KAGGLE_MODEL_PATH
    print(f"Using offline model: {MODEL_NAME}")
else:
    print(f"Using online model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ── Dataset ──
MAX_LEN = 256  # prompt + option combined

class MCQTransformerDataset(Dataset):
    """
    For each question, create 5 prompt-option pairs.
    Each pair: [CLS] prompt [SEP] option_text [SEP]
    """
    def __init__(self, df, is_test=False):
        self.df = df.reset_index(drop=True)
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])

        input_ids_list = []
        attention_mask_list = []
        token_type_ids_list = []

        for opt_col in LABELS:
            option_text = str(row[opt_col])
            encoded = tokenizer(
                prompt, option_text,
                max_length=MAX_LEN,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            input_ids_list.append(encoded['input_ids'].squeeze(0))
            attention_mask_list.append(encoded['attention_mask'].squeeze(0))
            if 'token_type_ids' in encoded:
                token_type_ids_list.append(encoded['token_type_ids'].squeeze(0))

        input_ids = torch.stack(input_ids_list)       # (5, MAX_LEN)
        attention_mask = torch.stack(attention_mask_list)  # (5, MAX_LEN)

        result = {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
        }
        if token_type_ids_list:
            result['token_type_ids'] = torch.stack(token_type_ids_list)

        if not self.is_test:
            result['labels'] = torch.tensor(label2id[row['answer']], dtype=torch.long)

        return result

# ── Model ──
class DeBERTaMCQ(nn.Module):
    """
    DeBERTa-v3 for Multiple Choice QA.
    Encodes each (prompt, option) pair independently,
    then uses a linear head to score each option.
    """
    def __init__(self, model_name, dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size  # 768 for base

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1)
        )

        # Multi-sample dropout for regularization
        self.ms_dropout = nn.ModuleList([nn.Dropout(0.1 * i) for i in range(1, 6)])

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        """
        input_ids:      (B, 5, L)
        attention_mask:  (B, 5, L)
        returns:         (B, 5) logits
        """
        B, num_choices, L = input_ids.shape

        # Flatten for backbone: (B*5, L)
        flat_input = input_ids.view(-1, L)
        flat_mask  = attention_mask.view(-1, L)
        flat_ttids = token_type_ids.view(-1, L) if token_type_ids is not None else None

        kwargs = {'input_ids': flat_input, 'attention_mask': flat_mask}
        if flat_ttids is not None:
            kwargs['token_type_ids'] = flat_ttids

        outputs = self.backbone(**kwargs)

        # Use [CLS] token representation — cast to float32 since DeBERTa-v3 outputs FP16
        cls_output = outputs.last_hidden_state[:, 0, :].float()  # (B*5, H)

        # Multi-sample dropout: average predictions from multiple dropout masks
        logits = torch.zeros(cls_output.size(0), 1, device=cls_output.device)
        for drop in self.ms_dropout:
            logits += self.classifier(drop(cls_output))
        logits = logits / len(self.ms_dropout)  # (B*5, 1)

        logits = logits.view(B, num_choices)  # (B, 5)
        return logits

# ── Training Config ──
N_FOLDS = 5
EPOCHS = 4
BATCH_SIZE = 4       # Small due to 5 options × MAX_LEN
ACCUM_STEPS = 4      # Effective batch = 16
LR = 1e-5
WARMUP_RATIO = 0.1

# Use Kaggle Secrets (add WANDB_API_KEY in Add-ons → Secrets)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    wandb_key = secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_key)
except Exception:
    wandb.login()  # fallback: interactive prompt
wandb.init(project="smart-mcq-solver", name="model3-deberta-v3",
           config={"model": "DeBERTa-v3-base", "epochs": EPOCHS,
                   "lr": LR, "batch_size": BATCH_SIZE, "max_len": MAX_LEN})

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
y_all = train['answer'].map(label2id).values
oof_probs = np.zeros((len(train), 5))
test_probs = np.zeros((len(test), 5))

test_ds = MCQTransformerDataset(test, is_test=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2)

for fold, (tr_idx, val_idx) in enumerate(skf.split(train, y_all)):
    print(f"\n{'='*60}")
    print(f"FOLD {fold+1}/{N_FOLDS}")
    print(f"{'='*60}")

    train_ds = MCQTransformerDataset(train.iloc[tr_idx])
    val_ds   = MCQTransformerDataset(train.iloc[val_idx])
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2)

    model = DeBERTaMCQ(MODEL_NAME, dropout=0.15).to(DEVICE)

    # Layer-wise learning rate decay
    backbone_params = []
    head_params = []
    for name, param in model.named_parameters():
        if 'backbone' in name:
            backbone_params.append(param)
        else:
            head_params.append(param)

    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': LR},
        {'params': head_params, 'lr': LR * 10}
    ], weight_decay=0.01)

    total_steps = len(train_loader) * EPOCHS // ACCUM_STEPS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    best_map3 = 0

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            token_type_ids = batch.get('token_type_ids')
            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            logits = model(input_ids, attention_mask, token_type_ids)
            loss = criterion(logits, labels) / ACCUM_STEPS

            loss.backward()
            train_loss += loss.item() * ACCUM_STEPS

            if (step + 1) % ACCUM_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

        # Validation
        model.eval()
        val_preds = []
        val_labels_list = []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                token_type_ids = batch.get('token_type_ids')
                if token_type_ids is not None:
                    token_type_ids = token_type_ids.to(DEVICE)

                logits = model(input_ids, attention_mask, token_type_ids)
                probs = F.softmax(logits.float(), dim=1).cpu().numpy()
                val_preds.append(probs)
                val_labels_list.append(batch['labels'].numpy())

        val_preds = np.concatenate(val_preds)
        val_labels_arr = np.concatenate(val_labels_list)

        val_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in val_preds]
        val_true = [id2label[y] for y in val_labels_arr]
        epoch_map3 = map_at_3(val_true, val_top3)

        avg_loss = train_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val MAP@3: {epoch_map3:.4f}")
        wandb.log({"fold": fold+1, "epoch": epoch+1,
                   "train_loss": avg_loss, "val_map3": epoch_map3,
                   "lr": optimizer.param_groups[0]['lr']})

        if epoch_map3 > best_map3:
            best_map3 = epoch_map3
            best_val_preds = val_preds.copy()
            torch.save(model.state_dict(), f"deberta_fold{fold}.pt")

    print(f"Fold {fold+1} Best MAP@3: {best_map3:.4f}")
    oof_probs[val_idx] = best_val_preds

    # Test inference with best checkpoint
    model.load_state_dict(torch.load(f"deberta_fold{fold}.pt", weights_only=True))
    model.eval()
    fold_test_preds = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            token_type_ids = batch.get('token_type_ids')
            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(DEVICE)
            logits = model(input_ids, attention_mask, token_type_ids)
            probs = F.softmax(logits.float(), dim=1).cpu().numpy()
            fold_test_preds.append(probs)
    test_probs += np.concatenate(fold_test_preds) / N_FOLDS

    del model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()

# Overall OOF MAP@3
oof_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in oof_probs]
oof_true = [id2label[y] for y in y_all]
overall_map3 = map_at_3(oof_true, oof_top3)
print(f"\n=== Model 3 (DeBERTa-v3) Overall OOF MAP@3: {overall_map3:.4f} ===")
wandb.log({"overall_map3": overall_map3})
wandb.finish()

np.save("deberta_test_probs.npy", test_probs)
np.save("deberta_oof_probs.npy", oof_probs)

# Submission
test_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in test_probs]
sub = pd.DataFrame({
    "ID": test["id"],
    "Prediction": [" ".join(t) for t in test_top3]
})
sub.to_csv("submission_deberta.csv", index=False)
print("Model 3 submission saved to submission_deberta.csv")
print(sub.head(10))


Device: cuda
Using online model: microsoft/deberta-v3-base


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260605_121324-udjthp1m
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model3-deberta-v3
wandb: ⭐️ View project at https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver/runs/udjthp1m


...

Epoch 1/4 | Loss: nan | Val MAP@3: 0.3817
Epoch 2/4 | Loss: nan | Val MAP@3: 0.3817
Epoch 3/4 | Loss: nan | Val MAP@3: 0.3817
Epoch 4/4 | Loss: nan | Val MAP@3: 0.3817
Fold 5 Best MAP@3: 0.3817


wandb: updating run metadata



=== Model 3 (DeBERTa-v3) Overall OOF MAP@3: 0.3835 ===


wandb: uploading wandb-summary.json; uploading config.yaml; uploading output.log
wandb: 
wandb: Run history:
wandb:        epoch ▁▃▆█▁▃▆█▁▃▆█▁▃▆█▁▃▆█
wandb:         fold ▁▁▁▁▃▃▃▃▅▅▅▅▆▆▆▆████
wandb:           lr █▅▂▁█▅▂▁█▅▂▁█▅▂▁█▅▂▁
wandb: overall_map3 ▁
wandb:     val_map3 ████████████▆▆▆▆▁▁▁▁
wandb:           +1 ...
wandb: 
wandb: Run summary:
wandb:        epoch 4
wandb:         fold 5
wandb:           lr 0
wandb: overall_map3 0.3835
wandb:   train_loss nan
wandb:     val_map3 0.38167
wandb: 
wandb: 🚀 View run model3-deberta-v3 at: https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver/runs/udjthp1m
wandb: ⭐️ View project at: https://wandb.ai/24f1002825-iit-madras-foundation/smart-mcq-solver
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260605_121324-udjthp1m/logs


Model 3 submission saved to submission_deberta.csv
   ID Prediction
0   1      A B C
1   2      A B C
2   3      A B C
3   4      A B C
4   5      A B C
5   6      A B C
6   7      A B C
7   8      A B C
8   9      A B C
9  10      A B C


In [6]:
"""
============================================================
SMART MCQ SOLVER CHALLENGE — ENSEMBLE & FINAL SUBMISSION
============================================================
Combines predictions from all 3 models with optimized weights.
Run AFTER all 3 model scripts have completed.
============================================================
"""

import pandas as pd
import numpy as np
import os
from scipy.optimize import minimize

# ── MAP@3 ──
def map_at_3(y_true, y_pred_top3):
    scores = []
    for true, preds in zip(y_true, y_pred_top3):
        score = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return np.mean(scores)

# ── Data ──
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "."

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
LABELS = ['A', 'B', 'C', 'D', 'E']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}
y_all = train['answer'].map(label2id).values

# ── Load OOF and Test Predictions ──
print("Loading model predictions...")

# Check which models have predictions
model_oof = {}
model_test = {}

if os.path.exists("lgbm_oof_probs.npy"):
    model_oof['lgbm'] = np.load("lgbm_oof_probs.npy")
    model_test['lgbm'] = np.load("lgbm_test_probs.npy")
    print("  ✓ LightGBM predictions loaded")

if os.path.exists("textcnn_oof_probs.npy"):
    model_oof['textcnn'] = np.load("textcnn_oof_probs.npy")
    model_test['textcnn'] = np.load("textcnn_test_probs.npy")
    print("  ✓ TextCNN predictions loaded")

if os.path.exists("deberta_oof_probs.npy"):
    model_oof['deberta'] = np.load("deberta_oof_probs.npy")
    model_test['deberta'] = np.load("deberta_test_probs.npy")
    print("  ✓ DeBERTa predictions loaded")

print(f"\nTotal models available: {len(model_oof)}")

# ── Individual Model Scores ──
print("\n--- Individual Model OOF MAP@3 ---")
for name, oof_probs in model_oof.items():
    top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in oof_probs]
    true = [id2label[y] for y in y_all]
    score = map_at_3(true, top3)
    print(f"  {name}: {score:.4f}")

# ── Optimize Ensemble Weights ──
if len(model_oof) > 1:
    print("\n--- Optimizing Ensemble Weights ---")
    model_names = list(model_oof.keys())
    oof_arrays = [model_oof[n] for n in model_names]

    def neg_map3(weights):
        """Negative MAP@3 for scipy minimization."""
        w = np.array(weights)
        w = w / w.sum()  # normalize
        blended = sum(w_i * oof for w_i, oof in zip(w, oof_arrays))
        top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in blended]
        true = [id2label[y] for y in y_all]
        return -map_at_3(true, top3)

    # Grid search first for rough weights, then refine
    best_score = -1
    best_weights = None

    # Quick grid search
    n_models = len(model_names)
    if n_models == 2:
        for w1 in np.arange(0, 1.05, 0.05):
            w2 = 1 - w1
            score = -neg_map3([w1, w2])
            if score > best_score:
                best_score = score
                best_weights = [w1, w2]
    elif n_models == 3:
        for w1 in np.arange(0, 1.05, 0.1):
            for w2 in np.arange(0, 1.05 - w1, 0.1):
                w3 = 1 - w1 - w2
                if w3 < 0:
                    continue
                score = -neg_map3([w1, w2, w3])
                if score > best_score:
                    best_score = score
                    best_weights = [w1, w2, w3]

    # Refine with scipy
    from scipy.optimize import minimize
    result = minimize(
        neg_map3, best_weights,
        method='Nelder-Mead',
        options={'maxiter': 1000, 'xatol': 0.001}
    )
    final_weights = np.array(result.x)
    final_weights = final_weights / final_weights.sum()
    final_score = -result.fun

    print(f"\nOptimized weights:")
    for name, w in zip(model_names, final_weights):
        print(f"  {name}: {w:.4f}")
    print(f"Ensemble MAP@3: {final_score:.4f}")

    # ── Generate Ensemble Test Predictions ──
    test_arrays = [model_test[n] for n in model_names]
    ensemble_test = sum(w * t for w, t in zip(final_weights, test_arrays))

else:
    # Single model
    name = list(model_oof.keys())[0]
    ensemble_test = model_test[name]
    print(f"\nUsing single model: {name}")

# ── Final Submission ──
test_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in ensemble_test]
sub = pd.DataFrame({
    "ID": test["id"],
    "Prediction": [" ".join(t) for t in test_top3]
})
sub.to_csv("submission.csv", index=False)
print(f"\n{'='*50}")
print("FINAL SUBMISSION saved to submission.csv")
print(f"{'='*50}")
print(f"Shape: {sub.shape}")
print(f"\nFirst 10 rows:")
print(sub.head(10))

# Sanity check
assert len(sub) == len(test), f"Row count mismatch: {len(sub)} vs {len(test)}"
assert list(sub.columns) == ['ID', 'Prediction'], f"Column mismatch: {list(sub.columns)}"
assert all(len(p.split()) == 3 for p in sub['Prediction']), "Some predictions don't have 3 answers"
print("\n✓ All sanity checks passed!")


Loading model predictions...
  ✓ LightGBM predictions loaded
  ✓ TextCNN predictions loaded
  ✓ DeBERTa predictions loaded

Total models available: 3

--- Individual Model OOF MAP@3 ---
  lgbm: 1.0000
  textcnn: 1.0000
  deberta: 0.3835

--- Optimizing Ensemble Weights ---

Optimized weights:
  lgbm: 0.0000
  textcnn: 0.0000
  deberta: 1.0000
Ensemble MAP@3: 0.3835

FINAL SUBMISSION saved to submission.csv
Shape: (500, 2)

First 10 rows:
   ID Prediction
0   1      A B C
1   2      A B C
2   3      A B C
3   4      A B C
4   5      A B C
5   6      A B C
6   7      A B C
7   8      A B C
8   9      A B C
9  10      A B C

✓ All sanity checks passed!
